In [29]:
import os
from pathlib import Path
import cv2
import numpy as np
from tqdm import tqdm
import mediapipe as mp

mp_holistic = mp.solutions.holistic

VIDEO_INPUT = r'E:\Datasets\wlasl300_dataset\WLASL300'
VIDEO_OUTPUT = r'E:\Balanced_20_Frames_Augmented\NPY'

In [30]:
"""Extraction of Landmarks , without facemesh"""

# ---------- 

def lm_to_np(lms, n):
    """Convert landmarks to (n,3). If missing, return zeros."""
    if lms is None:
        return np.zeros((n, 3), dtype=np.float32)
    return np.array([[lm.x, lm.y, lm.z] for lm in lms.landmark], dtype=np.float32)


def extract_75(results):
    """[pose(33) | left(21) | right(21)] => (75,3)"""
    pose = lm_to_np(results.pose_landmarks, 33)
    left = lm_to_np(results.left_hand_landmarks, 21)
    right = lm_to_np(results.right_hand_landmarks, 21)
    return np.concatenate([pose, left, right], axis=0)


def normalize_landmarks(frame_lm):
    """
    Normalize w.r.t shoulder center for invariance to position.
    Uses pose landmarks:
      left_shoulder = 11, right_shoulder = 12
    """
    left_sh = frame_lm[11]
    right_sh = frame_lm[12]
    center = (left_sh + right_sh) / 2.0
    frame_lm -= center
    return frame_lm


# --------

def video_to_npy(video_path, output_path, normalize=True):
    cap = cv2.VideoCapture(str(video_path))
    frames = []

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        refine_face_landmarks=False
    ) as holistic:

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = holistic.process(rgb)

            lm = extract_75(results)          # (75,3)

            if normalize:
                lm = normalize_landmarks(lm)

            frames.append(lm)

    cap.release()

    if len(frames) == 0:
        print(f"Skipped (no frames): {video_path}")
        return

    arr = np.stack(frames).astype(np.float16)  # (T,75,3)
    np.save(output_path, arr)
    print(f"Saved {output_path}  Shape: {arr.shape}")


# --------

def convert_dataset(video_root, npy_root):
    video_root = Path(video_root)
    npy_root = Path(npy_root)

    if video_root.exists():
        print("vid_path exist")
    else:
        print("vid path dont exist")

    videos = list(video_root.rglob("*.mp4"))
    
    print(len(videos))

    for vid in tqdm(videos):
        rel = vid.relative_to(video_root).with_suffix(".npy")
        out_path = npy_root / rel
        out_path.parent.mkdir(parents=True, exist_ok=True)

        video_to_npy(vid, out_path)


# -------

if __name__ == "__main__":
    convert_dataset(VIDEO_INPUT, VIDEO_OUTPUT)


vid_path exist
3565


  0%|          | 1/3565 [00:05<5:21:41,  5.42s/it]

Saved E:\Balanced_20_Frames_Augmented\NPY\1\00412.npy  Shape: (57, 75, 3)


  0%|          | 2/3565 [00:14<7:33:09,  7.63s/it]

Saved E:\Balanced_20_Frames_Augmented\NPY\1\00414.npy  Shape: (105, 75, 3)


...

Saved E:\Balanced_20_Frames_Augmented\NPY\99\17665.npy  Shape: (97, 75, 3)


100%|█████████▉| 3564/3565 [7:28:00<00:08,  8.02s/it]

Saved E:\Balanced_20_Frames_Augmented\NPY\99\68040.npy  Shape: (76, 75, 3)


100%|██████████| 3565/3565 [7:28:08<00:00,  7.54s/it]

Saved E:\Balanced_20_Frames_Augmented\NPY\99\70250.npy  Shape: (101, 75, 3)


In [ ]:
import numpy as np
import cv2

POSE_CONNECTIONS = [
    (11,12),(11,13),(13,15),(12,14),(14,16),
    (11,23),(12,24),(23,24),
    (23,25),(25,27),(27,29),(29,31),
    (24,26),(26,28),(28,30),(30,32)
]

HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (5,9),(9,10),(10,11),(11,12),
    (9,13),(13,14),(14,15),(15,16),
    (13,17),(17,18),(18,19),(19,20),
    (0,17)
]


def compute_global_transform(data, size=720, pad=80):
    """
    Compute ONE transform for the whole sequence
    so skeleton stays centered.
    """
    all_pts = data[:, :, :2].reshape(-1, 2)

    min_xy = all_pts.min(axis=0)
    max_xy = all_pts.max(axis=0)

    span = max_xy - min_xy
    span[span == 0] = 1e-6

    scale = (size - 2*pad) / max(span)

    # Centering offset
    center_after_scale = (min_xy + max_xy) / 2 * scale
    canvas_center = np.array([size/2, size/2])

    offset = canvas_center - center_after_scale

    return scale, offset


def transform_points(points, scale, offset):
    pts = points[:, :2] * scale + offset
    return pts.astype(int)


def draw(img, pts, connections, color):
    for i, j in connections:
        cv2.line(img, tuple(pts[i]), tuple(pts[j]), color, 2)


def npy_to_skeleton_video(npy_path, out_video, fps=30, size=720):
    data = np.load(npy_path).astype(np.float32)  # (T,75,3)

    scale, offset = compute_global_transform(data, size=size)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(out_video, fourcc, fps, (size, size))

    for frame in data:
        canvas = np.zeros((size, size, 3), dtype=np.uint8)

        pts2d = transform_points(frame, scale, offset)

        pose = pts2d[0:33]
        left = pts2d[33:54]
        right = pts2d[54:75]

        draw(canvas, pose, POSE_CONNECTIONS, (0,255,0))
        draw(canvas, left, HAND_CONNECTIONS, (255,0,0))
        draw(canvas, right, HAND_CONNECTIONS, (0,0,255))

        writer.write(canvas)

    writer.release()
    print(f"Saved skeleton video → {out_video}")


In [ ]:
# npy_to_skeleton_video(
#     r"E:\Balanced_20_Frames_Augmented\NPY\1\00415.npy",
#     "preview.mp4",
#     fps=30
# )


Saved skeleton video → preview.mp4
